# Trees and Forests

In these notes we go over decision trees and random forests, along with associated methods. The motivation for this, as a start, is to build a model for the Kaggle Titanic Survival ML project.



## 1. Decision Trees

A decision tree is a supervised learning algorithm, which may be used both for classification and for regression tasks. It effectively works like a flowchart, making step-by-step decisions as one moves through the hierarchical tree structure.

In these notes we will focus on a supervised classification task, i.e. the classification of a single discrete target variable. 


### Basic idea

Each internal node performs a Boolean tests on an input feature (which in general may have more than two options, which can be converted into a series of Boolean tests). We move through the trees along edges until we reach a leaf, upon which we return the classification on that leaf.

Overall, the tree contains:
- Root note: the first attribute test that every classification task passes through,
- Internal nodes: further attribute tests reached as we move down the tree,
- Branches (or edges): representing attribute values that have been decided at the nodes,
- Leaf nodes: representing the final decision or prediction that we arrive at.

### Attribute selection measures

For each dataset, there are many different decision trees we could build.
How do we choose which features we should test at each step? This is the essential question we run into when designing and training a decision tree. Further, given a number of features, we could train a very detailed decision tree, however this runs the risk of over-fitting. One way to avoid over-fitting is to constrain our model to be a simple one, in this case this corresponds to minimizing the number of features we have to test by learning a small and shallow tree.

#### Information gain (entropy)

This is used to tell us how useful a question (or feature) is for splitting data into groups, measuring how much the uncertainty decreases after the split. Essentially, we want to choose a feature that reduces our uncertainty as much as possible at a given node.


To accomplish this, we can use the *entropy*. For a dataset $S$ with $c$ target classes, the entropy $H(S)$ is calculated as
$$
H(S) = -\sum_{i=1}^{c}p_i \log_2(p_i)
$$
where
- $p_i$ is the proportion (i.e. the probability) of examples belonging to class $i$ of the dataset,
- if a node is completely pure (i.e. all instances belong to one class), then $H(S)=0$,
- if classes a split evenly (e.g. a 50/50 choice), then $H(S)=1$.

The *Information Gain* $I(S,A)$ of a feature $A$ relative to the dataset $S$ is the difference between initial entropy and the weighted average entropy of the child nodes after the split:
$$
I(S,A) = H(S) - \sum_{v\in \mathrm{Values}(A)}\frac{|S_v|}{|S|}H(S_v)
$$
where
- $\mathrm{Values}(A)$ is the set of all possible outcomes for feature $A$,
- $S_v$ is the subset of $S$ where feature $A$ has the value $v$,
- $|S_v|/|S|$ acts as the weight, representing the fraction of instances that flow into that specific branch.

##### Example: Titanic Survivors

We are predicting a binary target, $y\in \{0,1\}$, where Did Not Survive = 0, Survived = 1. At the root of the tree, all of our training observations are mixed together, e.g. our parent node contains:
$$
[1, 1, 1, 1, 0, 0, 0, 0]
$$
This is a 50/50 split between our two classes, meaning we have the maximum amount of uncertainty. If a feature allows us to split this into
$$
\mathrm{Left}: [1,1,1,1]
$$
$$
\mathrm{Right}: [0,0,0,0]
$$
then each child of the split is perfectly pure, i.e. we have gained the maximum amount of information. This would be the best possible split, and we use information gain to measure exactly how good the split it.


As a more detailed example, let's say we have 10 passengers:
|Sex|Survived|
|-|-|
|Female|1|
|Female|1|
|Female|1|
|Female|1|
|Male|1|
|Male|0|
|Male|0|
|Male|0|
|Male|0|
|Male|0|

We first split on Sex, and find that the four women all survive (i.e. Survived = 1). This means that following the split, the Female child is perfectly pure, meaning we have the maximum amount of information gain, and can stop this branch at this point, assigning Survived=1 to the leaf.

For the Male child, we have 6 men, one with Survived=1. This means that the entropy is
$$
H(\mathrm{Male}) = -\frac{1}{6} \log_2 \frac{1}{6} - \frac{5}{6}\log_2\frac{5}{6} \simeq 0.65.
$$
The entropy after the split, noting that we have reduced the entropy of the Female split to 0, is
$$
H_{\mathrm{after}} = \frac{4}{10}(0) + \frac{6}{10}(0.65) \simeq 0.39
$$
Since the distribution at the start was 5 survivors and 5 non-survivors, the entropy was 1, therefore the information gain is:
$$
I = 1 - 0.39 \simeq 0.61,
$$
which is a substantial amount of information gained from this split.

Note that the female split arrives at a pure distribution, meaning that the tree stops at this leaf. However, the male child does not arrive at a perfectly pure distribution, so we could therefore continue this branch until we reach a pure distribution.

### Stopping conditions
This brings us to stopping conditions. The main ones are:
- The node is pure (e.g. $H=0$),
- The maximum depth is reached (set by max_depth)
- Too few samples remain to split (set by min_samples_split)
- A split would make a child too small (set by min_samples_leaf)
- The split does not improve impurity enough (set by min_impurity_decrease)
- A leaf-count limit is reached (set by max_leaf_nodes)


Note that the decision tree algorithm is **greedy**, meaning it bases its decision on what it can learn right now, rather than optimising the *entire* decision tree, which could be very computationally intensive.


## Gini Index

There is an alternative method for measuring impurity, known as the Gini Index. This is a measure of how often a randomly chosen element would be incorrectly identified, meaning an attribute with a lower Gini index should be preferred. For example, if we have a group of people who all survived, (i.e. 100% Survived=1), then the Gini index is 0, indicating perfect purity.

The Gini Index measure is given by:
$$
\mathrm{Gini} = 1 - \sum_{i=1}^{n} p_i^2
$$
and works in the same overall way as the entropy detailed above.

## 2. Implementing using Scikit-Learn

Here, we focus on a binary classification problem, in particular the Titanic Surivival Kaggle problem.


The basic syntax is:
```python
class sklearn.tree.DecisionTreeClassifier(
    *,
    criterion='entropy',
    splitter='best',
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    min_weight_fraction_leaf=0.0,
    max_features=None,
    random_state=None,
    max_leaf_nodes=None,
    min_impurity_decrease=0.0,
    class_weight=None,
    ccp_alpha=0.0,
    monotonic_cst=None
)
```
**Parameters:**
- criterion - metric used to evaluate split (entropy, gini)
- splitter: strategy for choosing splits (best, random)
- max_features: number of features considered for each split
- max_epth: maximum depth of the tree
- min_samples_split: minimum samples required to split a node
- min_samples_leaf: minimum samples required at a leaf node
- max_leaf_nodes: maximum number of leaf nodes
- min_impurity_decrease: minimum impurity reduction required to split a node
- class_weight: balances class distribution by assigning weights
- ccp_alpha: controls pruning strength to reduce overfitting

1. We first import the relevent libraries
```python
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
```

2. We then split the dataset into training and testing:
```python
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=99
)
```

3. Using the DecisionTreeClassifier from sklearn.tree, we create an object for the decision tree classifier
```python
clf = DecisionTreeClassifier(random_state=1)
```

4. Training the model is accomplished using .fit:
```python
clf.fit(X_train, y_train)
```

5. We can then make use the trained model to make predictions based on the test data to assess its accuracy
```python
y_pred = clf.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy}")
```


6. Hyperparameter Tuning

We can use GridSearchCV to tune our hyperparameters, which for the decision tree are configureation settings that control how the decision tree model learns from data.
```python
from sklearn.model_selection import GridSearchCV

param_grid = {
    'max_depth': range(1, 10, 1),
    'min_samples_leaf': range(1, 20, 2),
    'min_samples_split': range(2, 20, 2),
    'criterion': ["entropy", "gini"]
}

tree = DecisionTreeClassifier(random_state=1)

grid_search = GridSearchCV(estimator=tree, param_grid=param_grid,
                           cv=5, verbose=True)

grid_search.fit(X_train, y_train)

print("best_accuracy", grid_search.best_score_)
print(grid_search.best_estimator_)
```

7. Finally, we can visualise the decision tree classifier. This is useful to interpret and comprehend the model's choices.

```python
from sklearn.tree import plot_tree
import matplotlib.pyplot as plt

tree_clf = grid_search.best_estimator_

plt.figure(figsize=(18, 15))
plot_tree(tree_clf, filled=True, feature_names=data.feature_names,
          class_names=data.target_names)
plt.show()
```


## 3. Random Forests

Random forests is a ML algorithm that uses many decision trees to make better predictions. As an **overview**, random forests:
- Create many random decision trees: using random parts of the data the algorithm creates many distinct decision trees
- Pick random features: when building each tree, it doesn't look at all the features at once, it picks a few at random and decides how to split the data. This helps each tree be distinct
- Each tree makes a prediction: every tree gives its own answer or prediction based on what it learned from its part of the data
- combine the predictions: for classification, the final answer is the category that most trees vote for (majority voting)
- why it works well: using random data and features for each tree helps avoid overfitting and makes the overall prediction more accurate and trustworthy

Important features:
- This helps show feature importance, telling you what features are most useful for making predictions, allowing for a better understanding of your data
- works well for large datasets without slowing down or losing accuracy
- can be used for both classification and regression

**Importantly:**
- each tree is independent of all other trees, helping reduce possible mistakes
- enough data is needed to ensure all trees are different and learn unique patterns


### Implementation (Classification)

This is taken from (https://www.geeksforgeeks.org/machine-learning/random-forest-algorithm-in-machine-learning/).

Implementation is on the titanic survival rate classification task.

The flow is:
- import libraries
- load the data
- remove rows with missing target values
- select features
- fill missing age values with the median
- split the data into training and testing
- train random forest model
- predict on the test data, check accuracy, and print a sample prediction result
```python
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import warnings
warnings.filterwarnings('ignore')

titanic_data = pd.read_csv('titanic.csv')

titanic_data = titanic_data.dropna(subset=['Survived'])

X = titanic_data[['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare']]
y = titanic_data['Survived']

X['Sex'] = X['Sex'].map({'female': 0, 'male': 1})
X['Age'] = X['Age'].fillna(X['Age'].median())

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42)
rf_classifier.fit(X_train, y_train)

y_pred = rf_classifier.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
classification_rep = classification_report(y_test, y_pred)

print(f"Accuracy: {accuracy:.2f}")
print("\nClassification Report:\n", classification_rep)

sample = X_test.iloc[0:1]
prediction = rf_classifier.predict(sample)

sample_dict = sample.iloc[0].to_dict()
print(f"\nSample Passenger: {sample_dict}")
print(f"Predicted Survival: {'Survived' if prediction[0] == 1 else 'Did Not Survive'}")
```


### Implementation (Regression)

House pricing regression task (from: https://www.geeksforgeeks.org/machine-learning/random-forest-algorithm-in-machine-learning/)

Workflow:
- load dataset
- separate features and target variable
- train/test split
- train random forest regressor
- predict house values on test data and evaluate using MSE and $R^2$
- print a sample prediction and compare it with the actual value

```python
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

california_housing = fetch_california_housing()
california_data = pd.DataFrame(california_housing.data, columns=california_housing.feature_names)
california_data['MEDV'] = california_housing.target

X = california_data.drop('MEDV', axis=1)
y = california_data['MEDV']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf_regressor = RandomForestRegressor(n_estimators=100, random_state=42)

rf_regressor.fit(X_train, y_train)

y_pred = rf_regressor.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

single_data = X_test.iloc[0].values.reshape(1, -1)
predicted_value = rf_regressor.predict(single_data)
print(f"Predicted Value: {predicted_value[0]:.2f}")
print(f"Actual Value: {y_test.iloc[0]:.2f}")

print(f"Mean Squared Error: {mse:.2f}")
print(f"R-squared Score: {r2:.2f}")
```